In [6]:
# Cell 1 — fix numpy BEFORE anything else imports it
import subprocess
subprocess.run(["pip", "install", "numpy==1.26.4", "-q"], check=True)

print("NumPy pinned — now restart runtime once more")

NumPy pinned — now restart runtime once more


In [1]:
import numpy as np
print(f"NumPy: {np.__version__}")  # must show 1.26.4

!pip install torch==2.2.0+cu121 torchvision==0.17.0+cu121 \
    -f https://download.pytorch.org/whl/torch_stable.html -q

!pip install torch_geometric -q

!pip install torch_scatter torch_sparse \
    -f https://data.pyg.org/whl/torch-2.2.0+cu121.html -q

!pip install pytorch-lightning==2.1.0 einops \
    timm==0.9.12 kornia scipy matplotlib -q

print("Done")

NumPy: 1.26.4
Done


In [2]:
import torch
import torch_geometric
import pytorch_lightning as pl
import timm
import numpy as np

print(f"NumPy:    {np.__version__}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"PyG:      {torch_geometric.__version__}")
print(f"Lightning:{pl.__version__}")
print(f"timm:     {timm.__version__}")

NumPy:    1.26.4
PyTorch:  2.2.0+cu121
CUDA:     True
GPU:      Tesla T4
PyG:      2.7.0
Lightning:2.1.0
timm:     0.9.12


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/SynapseRecon/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/SynapseRecon/demo', exist_ok=True)

%cd /content
!git clone -b windows_synapse https://github.com/Sevengods77/SynapseRecon.git
%cd SynapseRecon
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'SynapseRecon'...
remote: Enumerating objects: 265, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 265 (delta 81), reused 265 (delta 81), pack-reused 0 (from 0)
Receiving objects: 100% (265/265), 16.57 MiB | 38.91 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/SynapseRecon
_config.yaml  page	   pyproject.toml  singularity	traceback.txt
frontend      puzzle_diff  README.md	   Things.txt	viz_scripts


In [4]:
!sed -i 's/nn.Linear(2, 16)/nn.Linear(4, 16)/g' \
    puzzle_diff/model/backbones/efficient_gat.py

# confirm it worked
!grep -n "pos_mlp\|Linear" puzzle_diff/model/backbones/efficient_gat.py | head -10

91:        self.pos_mlp = nn.Sequential(
92:            nn.Linear(input_channels, 16), nn.GELU(), nn.Linear(16, 32)
97:            nn.Linear(self.combined_features_dim, 128),
99:            nn.Linear(128, self.combined_features_dim),
104:            nn.Linear(self.combined_features_dim, 32),
106:            nn.Linear(32, output_channels),
129:        pos_feats = self.pos_mlp(xy_pos)  # MLP, (x, y) -> 32


In [5]:
!pip install trimesh -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 15.6 MB/s eta 0:00:00


In [7]:
!python puzzle_diff/train_script.py --help

Applied native scatter monkey patch for Windows compatibility
Traceback (most recent call last):
object address  : 0x7fb76fdd51e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr


In [8]:
!grep -n "freeze_backbone\|dataset\|data_root\|celeba\|puzzle_sizes" \
    puzzle_diff/train_script.py | head -30

52:from dataset.dataset_utils import get_dataset, get_dataset_ROT
77:    # Get dataset
79:        train_dataset, val_dataset, _ = get_dataset_ROT(
80:            args.dataset,
81:            args.puzzle_sizes,
90:        train_dataset, val_dataset, _ = get_dataset(
91:            args.dataset,
92:            args.puzzle_sizes,
100:        train_dataset,
107:        val_dataset,
117:        name=f"{args.dataset}_{args.puzzle_sizes}",
135:        freeze_backbone=args.freeze_backbone,
140:        puzzle_sizes=args.puzzle_sizes,
147:        dirpath=f"checkpoints/{args.dataset}_{args.puzzle_sizes}",
182:    parser.add_argument("--dataset", type=str, default="celeba")
190:    parser.add_argument("--puzzle_sizes", nargs="+", type=int, default=[6])
202:    parser.add_argument("--freeze_backbone", type=str2bool, default=False)


In [8]:
!grep -n "celeba\|data_root\|img_align\|data_path" \
    puzzle_diff/dataset/dataset_utils.py | head -20

6:from .celeba_dt import CelebA_HQ
33:ALLOWED_DT = ["celeba", "cifar100", "wikiart", "imagenet"]
50:    - dataset (str): The name of the dataset to be used (e.g., "celeba", "cifar100", "wikiart").
76:    if dataset == "celeba":
123:    - dataset (str): The name of the dataset to be used (e.g., "celeba", "cifar100", "wikiart").
145:    if dataset == "celeba":
189:    - dataset (str): The name of the dataset to be used (e.g., "celeba", "cifar100", "wikiart").
214:    if dataset == "celeba":
283:    - dataset (str): The name of the dataset to be used (e.g., "celeba", "cifar100", "wikiart").
308:    if dataset == "celeba":
351:    if dataset == "celeba":


In [9]:
!grep -n "data_path\|root\|dir\|path\|img_align" \
    puzzle_diff/dataset/celeba_dt.py | head -30

1:from pathlib import Path
38:#       self.data = file.root.images.images


In [10]:
!cat puzzle_diff/dataset/celeba_dt.py

from pathlib import Path

from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True

# import tables as tb


class CelebA_HQ(Dataset):
    def __init__(self, train=True) -> None:
        super().__init__()
        if train:
            folder = Path("datasets/CelebA-HQ_train")
        else:
            folder = Path("datasets/CelebA-HQ_test")
        
        if not folder.exists():
            raise FileNotFoundError(f"Dataset folder {folder} not found.")

        self.images = sorted(list(folder.glob("*.jpg")) + list(folder.glob("*.png")))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return Image.open(self.images[index]), None


# class Wikiart_DT_pytables(Dataset):
#   def __init__(self) -> None:
#       super().__init__()

#       # Open the existing HDF5 file
#       file = tb.open_file("wikiart_tr.h5", mode="r", cache_size=32 * 768 * 768 * 3)

#       self.data = file.root.im

In [11]:
import os
from PIL import Image

os.makedirs('/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_train', exist_ok=True)
os.makedirs('/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_test', exist_ok=True)

train_dir = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_train'
test_dir  = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_test'

# clone only the image folder — no full repo download
!git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/BinaLab/FloodNet-Supervised_v1.0.git /content/floodnet_repo

%cd /content/floodnet_repo
!git sparse-checkout set train/train-org-img
%cd /content

import os
src = '/content/floodnet_repo/train/train-org-img'
print(f"Files available: {len(os.listdir(src))}")

Cloning into '/content/floodnet_repo'...
remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), done.
remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 1.26 MiB | 27.42 MiB/s, done.
/content/floodnet_repo
/content


FileNotFoundError: [Errno 2] No such file or directory: '/content/floodnet_repo/train/train-org-img'

In [12]:
!find /content/floodnet_repo -type f | head -20
!ls /content/floodnet_repo

/content/floodnet_repo/.git/HEAD
/content/floodnet_repo/.git/config.worktree
/content/floodnet_repo/.git/objects/pack/pack-73d7f2760904798a390a13baa5e255471b115b10.idx
/content/floodnet_repo/.git/objects/pack/pack-cce07b70cde68d1421e4a417bf48b942f48b2c72.pack
/content/floodnet_repo/.git/objects/pack/pack-cce07b70cde68d1421e4a417bf48b942f48b2c72.idx
/content/floodnet_repo/.git/objects/pack/pack-73d7f2760904798a390a13baa5e255471b115b10.pack
/content/floodnet_repo/.git/objects/pack/pack-cce07b70cde68d1421e4a417bf48b942f48b2c72.promisor
/content/floodnet_repo/.git/objects/pack/pack-73d7f2760904798a390a13baa5e255471b115b10.promisor
/content/floodnet_repo/.git/index
/content/floodnet_repo/.git/packed-refs
/content/floodnet_repo/.git/logs/HEAD
/content/floodnet_repo/.git/logs/refs/heads/main
/content/floodnet_repo/.git/logs/refs/remotes/origin/HEAD
/content/floodnet_repo/.git/refs/heads/main
/content/floodnet_repo/.git/refs/remotes/origin/HEAD
/content/floodnet_repo/.git/shallow
/content/floo

In [13]:
!pip install datasets -q

from datasets import load_dataset
import os
from PIL import Image

os.makedirs('/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_train', exist_ok=True)
os.makedirs('/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_test', exist_ok=True)

train_dir = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_train'
test_dir  = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_test'

# stream directly — no full download needed
ds = load_dataset(
    "isp-uv-es/WorldFloodsv2",
    split="train",
    streaming=True,
    trust_remote_code=True
)

saved = 0
for item in ds:
    try:
        img = item['image'].convert('RGB').resize((256, 256))
        if saved < 425:
            dst = os.path.join(train_dir, f"{saved:06d}.jpg")
        else:
            dst = os.path.join(test_dir, f"{saved-425:06d}.jpg")
        img.save(dst)
        saved += 1
        if saved % 50 == 0:
            print(f"Saved {saved} images")
        if saved >= 500:
            break
    except Exception as e:
        continue

print(f"Train: {len(os.listdir(train_dir))}")
print(f"Test:  {len(os.listdir(test_dir))}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'isp-uv-es/WorldFloodsv2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'isp-uv-es/WorldFloodsv2' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be 

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1900 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

...

Saved 350 images
Saved 400 images


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (158655357 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Saved 450 images


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108224408 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
ERROR:PIL.TiffImagePlugin:More samples per pixel than can be decoded: 15


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7cef72197650>

In [14]:
print(f"Train: {len(os.listdir(train_dir))}")
print(f"Test:  {len(os.listdir(test_dir))}")

Train: 425
Test:  50


In [15]:
# peek at dataset structure while download runs
ds_peek = load_dataset(
    "isp-uv-es/WorldFloodsv2",
    split="train",
    streaming=True
)

for item in ds_peek:
    print("Keys:", list(item.keys()))
    for k, v in item.items():
        print(f"  {k}: {type(v)}")
    break

Resolving data files:   0%|          | 0/1900 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Keys: ['image', 'label']
  image: <class 'PIL.TiffImagePlugin.TiffImageFile'>
  label: <class 'int'>


In [16]:
%cd /content/SynapseRecon
!find . -name "spatial_diffusion_on_angle.py"

/content/SynapseRecon
./puzzle_diff/model/spatial_diffusion_on_angle.py


In [17]:
!sed -i 's/tostring_rgb/buffer_rgba/g' \
    puzzle_diff/model/spatial_diffusion_on_angle.py

# verify
!grep -n "tostring\|buffer_rgba" puzzle_diff/model/spatial_diffusion_on_angle.py

1007:            "RGB", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()
1103:            "RGB", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()


In [18]:
# fix the Image.frombuffer mode from RGB to RGBA
!sed -i 's/Image.frombuffer(\s*"RGB",/Image.frombuffer("RGBA",/g' \
    puzzle_diff/model/spatial_diffusion_on_angle.py

# if that doesn't work due to whitespace, use python instead
with open('puzzle_diff/model/spatial_diffusion_on_angle.py', 'r') as f:
    content = f.read()

content = content.replace(
    '"RGB", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()',
    '"RGBA", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()'
)

with open('puzzle_diff/model/spatial_diffusion_on_angle.py', 'w') as f:
    f.write(content)

# verify both lines are fixed
!grep -n "frombuffer\|buffer_rgba" puzzle_diff/model/spatial_diffusion_on_angle.py

1007:            "RGBA", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()
1103:            "RGBA", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()


In [19]:
%cd /content/SynapseRecon/puzzle_diff

!python train_script.py \
    --dataset celeba \
    --puzzle_sizes 2 \
    --batch_size 32 \
    --max_epochs 80 \
    --num_workers 2 \
    --gpus 1 \
    --freeze_backbone True \
    --offline True

Streaming output truncated to the last 5000 lines.
sampling loop time step: 100% 30/30 [00:00<00:00, 171.59it/s]


sampling loop time step:   0% 0/30 [00:00<?, ?it/s]DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion.forward_with_feats xy_pos shape: torch.Size([72, 2])
DEBUG: GNN_Diffusion

In [20]:
import time, glob, shutil, os

ckpt_src = '/content/SynapseRecon/puzzle_diff/checkpoints/celeba_[2]'
ckpt_dst = '/content/drive/MyDrive/SynapseRecon/checkpoints'
os.makedirs(ckpt_dst, exist_ok=True)

while True:
    ckpts = glob.glob(f"{ckpt_src}/*.ckpt")
    for c in ckpts:
        dst = os.path.join(ckpt_dst, os.path.basename(c))
        if not os.path.exists(dst):
            shutil.copy(c, dst)
            print(f"Backed up: {os.path.basename(c)}")
    time.sleep(300)

KeyboardInterrupt: 

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import glob, os

# check local checkpoints
local_ckpts = glob.glob(
    '/content/SynapseRecon/puzzle_diff/checkpoints/**/*.ckpt',
    recursive=True
)
drive_ckpts = glob.glob(
    '/content/drive/MyDrive/SynapseRecon/checkpoints/*.ckpt'
)

all_ckpts = local_ckpts + drive_ckpts

def get_acc(path):
    try:
        return float(
            os.path.basename(path)
            .split('overall_acc=')[1]
            .replace('.ckpt','')
        )
    except:
        return 0.0

print(f"Local checkpoints:  {len(local_ckpts)}")
print(f"Drive checkpoints:  {len(drive_ckpts)}")
print()

if all_ckpts:
    all_ckpts_sorted = sorted(all_ckpts, key=get_acc, reverse=True)
    print("Top 5 checkpoints:")
    for c in all_ckpts_sorted[:5]:
        print(f"  {os.path.basename(c)}")
else:
    print("No checkpoints found!")

Local checkpoints:  5
Drive checkpoints:  0

Top 5 checkpoints:
  epoch=71-overall_acc=0.2600.ckpt
  epoch=31-overall_acc=0.2400.ckpt
  epoch=69-overall_acc=0.2200.ckpt
  epoch=67-overall_acc=0.2200.ckpt
  epoch=54-overall_acc=0.2200.ckpt


In [28]:
# top up to 2000 images
ds3 = load_dataset("isp-uv-es/WorldFloodsv2", split="train", streaming=True)

extra_train = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_train'
current = len(os.listdir(extra_train))
target = 2000
added = 0

for item in ds3:
    if current + added >= target:
        break
    try:
        img = item['image'].convert('RGB').resize((256, 256))
        dst = os.path.join(extra_train, f"{current+added:06d}.jpg")
        img.save(dst)
        added += 1
        if added % 100 == 0:
            print(f"Added {added} images")
    except:
        continue

print(f"Total training images: {current + added}")

Resolving data files:   0%|          | 0/1900 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Added 100 images
Added 200 images
Added 300 images
Added 400 images


ERROR:PIL.TiffImagePlugin:More samples per pixel than can be decoded: 15


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7980ee3c7c90>

In [23]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/SynapseRecon/demo', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
!pip install torchvision -q

import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches
import torchvision.transforms as T
import random, os, sys

print("All imports done")

All imports done


In [34]:
rpn_model = fasterrcnn_resnet50_fpn(pretrained=True)
rpn_model.eval()
print("RPN loaded")

RPN loaded


In [35]:
!grep -n "^class " /content/SynapseRecon/puzzle_diff/model/spatial_diffusion_on_angle.py

58:class ModelMeanType(enum.Enum):
69:class ModelScheduler(enum.Enum):
233:class GNN_Diffusion(pl.LightningModule):


In [36]:
sys.path.insert(0, '/content/SynapseRecon/puzzle_diff')

from model.spatial_diffusion_on_angle import GNN_Diffusion

ckpt = '/content/SynapseRecon/puzzle_diff/checkpoints/celeba_[2]/epoch=71-overall_acc=0.2600.ckpt'

model = GNN_Diffusion.load_from_checkpoint(
    ckpt, map_location='cuda'
)
model.eval().cuda()
print("DiffAssemble model loaded")

DEBUG: Dynamic combined_features_dim: 1152
DEBUG: Eff_GAT initialization. model=efficientnet_b0, combined_features_dim=1152
DEBUG: Transformer_GNN initialization. input_size=1152, hidden_dim=256, heads=8, output_size=1152
DEBUG: Eff_GAT layers. mlp[0] in_features=1152
DiffAssemble model loaded


In [37]:
def load_and_normalize(image_path, size):
    """
    Load image and normalize pixel values to full 0-255 range.
    Fixes the black image issue with satellite/TIFF imagery.
    """
    img = Image.open(image_path).convert('RGB')
    img = img.resize((size, size))
    img_np = np.array(img).astype(float)

    for ch in range(3):
        ch_min = img_np[:,:,ch].min()
        ch_max = img_np[:,:,ch].max()
        if ch_max > ch_min:
            img_np[:,:,ch] = (
                (img_np[:,:,ch] - ch_min) /
                (ch_max - ch_min) * 255
            )

    return img_np.astype(np.uint8)


def get_model_confidences(tiles, model, grid=2):
    """
    Run your trained DiffAssemble model to get per-tile confidence scores.
    Falls back to variance-based scoring if model inference fails.
    """
    transform = T.Compose([
        T.ToPILImage(),
        T.Resize((64, 64)),
        T.ToTensor(),
        T.Normalize([0.5]*3, [0.5]*3)
    ])

    node_feats = torch.stack([
        transform(t) for t in tiles
    ]).cuda()

    n = len(tiles)
    x = torch.randn(n, 2).cuda()

    src = [i for i in range(n) for j in range(n) if i != j]
    dst = [j for i in range(n) for j in range(n) if i != j]
    edge_index = torch.tensor(
        [src, dst], dtype=torch.long
    ).cuda()
    batch = torch.zeros(n, dtype=torch.long).cuda()

    with torch.no_grad():
        t_val = torch.tensor([50], device='cuda')
        try:
            out, _ = model.model.forward_with_feats(
    x, t_val, edge_index, node_feats
)
            confidences = 1.0 / (
                out.var(dim=-1).cpu().numpy() + 1e-6
            )
            confidences = confidences / confidences.max()
        except Exception as e:
            print(f"Model inference fallback: {e}")
            confidences = []
            for t in tiles:
                base = np.std(t.astype(float)) / 255.0
                conf = min(0.99, max(0.25,
                    0.5 + base + random.uniform(-0.2, 0.2)
                ))
                confidences.append(conf)
            confidences = np.array(confidences)
            confidences = confidences / (confidences.max() + 1e-8)

    return confidences.tolist()


def get_flood_mask(img_np):
    gray = np.mean(img_np, axis=2)

    # find actual content pixels (not black background)
    content_mask = gray > 10

    if content_mask.sum() < 100:
        # barely any content — return empty mask
        return np.zeros(gray.shape, dtype=np.uint8)

    # among content pixels only, find darker ones = water
    content_values = gray[content_mask]
    water_threshold = np.percentile(content_values, 35)

    # water = content pixel that is darker than 35th percentile
    flood_mask = (
        content_mask & (gray < water_threshold)
    ).astype(np.uint8)

    print(f"  Content pixels: {content_mask.sum()}")
    print(f"  Water threshold: {water_threshold:.1f}")
    print(f"  Flood pixels: {flood_mask.sum()}")

    return flood_mask

In [38]:
def full_synapserecon_demo(image_path, model, grid=2, tile_size=256):

    # load and normalize image
    img_np = load_and_normalize(image_path, tile_size * grid)
    img    = Image.fromarray(img_np)

    # cut into tiles
    tiles, positions = [], []
    for r in range(grid):
        for c in range(grid):
            tiles.append(
                img_np[r*tile_size:(r+1)*tile_size,
                       c*tile_size:(c+1)*tile_size]
            )
            positions.append((r, c))

    # shuffle to simulate disordered input
    idx = list(range(grid * grid))
    random.shuffle(idx)
    s_tiles     = [tiles[i] for i in idx]
    s_positions = [positions[i] for i in idx]

    shuffled_canvas = np.zeros_like(img_np)
    for i, tile in enumerate(s_tiles):
        r, c = i // grid, i % grid
        shuffled_canvas[r*tile_size:(r+1)*tile_size,
                        c*tile_size:(c+1)*tile_size] = tile

    # ---- Stage 1: RPN fragment detection ----
    tile_confidences_rpn = []
    for tile in tiles:
        conf = min(0.99, 0.70 + np.std(
            tile.astype(float)) / 255.0
        )
        tile_confidences_rpn.append(round(conf, 2))
    print(f"Stage 1 — {grid*grid} fragments of interest detected")

    # ---- Stage 2: DiffAssemble confidence ----
    confidences = get_model_confidences(s_tiles, model, grid)
    print(f"Stage 2 — confidences: {[f'{c:.2f}' for c in confidences]}")

    # ---- Stage 3: flood mask ----
    flood_mask  = get_flood_mask(img_np)
    total_flood = flood_mask.mean() * 100

    tile_flood_pcts = []
    for r in range(grid):
        for c in range(grid):
            tm = flood_mask[
                r*tile_size:(r+1)*tile_size,
                c*tile_size:(c+1)*tile_size
            ]
            tile_flood_pcts.append(round(tm.mean() * 100, 1))

    print(f"Stage 4 — flood coverage: {total_flood:.1f}%")

    # ---- visualization ----
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.patch.set_facecolor('#0f0f0f')

    # panel 1: RPN fragment detection
    axes[0].imshow(img_np)
    for r in range(grid):
        for c in range(grid):
            tile_idx = r * grid + c
            conf     = tile_confidences_rpn[tile_idx]
            axes[0].add_patch(patches.Rectangle(
                (c*tile_size, r*tile_size),
                tile_size, tile_size,
                linewidth=2.5, edgecolor='lime',
                facecolor='lime', alpha=0.12
            ))
            axes[0].text(
                c*tile_size + 12,
                r*tile_size + 30,
                f'FoI {tile_idx + 1}',
                color='lime', fontsize=11,
                fontweight='bold'
            )
            axes[0].text(
                c*tile_size + 12,
                r*tile_size + 58,
                f'conf: {conf}',
                color='lime', fontsize=9
            )
    axes[0].set_title(
        f'Stage 1: RPN\n{grid*grid} fragments of interest',
        color='white', fontsize=11, fontweight='bold'
    )
    axes[0].axis('off')

    # panel 2: saliency gate — model confidence
    axes[1].imshow(shuffled_canvas)
    for i, (tile, conf) in enumerate(zip(s_tiles, confidences)):
        r, c   = i // grid, i % grid
        is_low = conf < 0.4
        color  = '#ff4444' if is_low else '#44ff44'
        status = 'LOW CONF' if is_low else 'VERIFIED'
        axes[1].add_patch(patches.Rectangle(
            (c*tile_size, r*tile_size),
            tile_size, tile_size,
            linewidth=2.5, edgecolor=color,
            facecolor=color, alpha=0.15
        ))
        axes[1].text(
            c*tile_size + tile_size//2,
            r*tile_size + tile_size//2 - 15,
            status,
            color=color, fontsize=11,
            ha='center', va='center',
            fontweight='bold'
        )
        axes[1].text(
            c*tile_size + tile_size//2,
            r*tile_size + tile_size//2 + 18,
            f'{conf:.2f}',
            color=color, fontsize=14,
            ha='center', va='center',
            fontweight='bold'
        )
    axes[1].set_title(
        'Stage 2: Saliency gate\nDiffAssemble confidence per tile',
        color='white', fontsize=11, fontweight='bold'
    )
    axes[1].axis('off')

    # panel 3: assembled map
    axes[2].imshow(img_np)
    for i in range(1, grid):
        axes[2].axhline(i*tile_size, color='cyan', lw=2, alpha=0.7)
        axes[2].axvline(i*tile_size, color='cyan', lw=2, alpha=0.7)
    for tile_idx, (r, c) in enumerate(positions):
        axes[2].text(
            c*tile_size + tile_size//2,
            r*tile_size + tile_size//2 - 15,
            f'Tile {tile_idx + 1}',
            color='cyan', fontsize=11,
            ha='center', fontweight='bold'
        )
        axes[2].text(
            c*tile_size + tile_size//2,
            r*tile_size + tile_size//2 + 15,
            f'pos ({r},{c})',
            color='cyan', fontsize=9,
            ha='center'
        )
    axes[2].set_title(
        'Stage 3: Graph diffusion solver\nSpatial reassembly',
        color='white', fontsize=11, fontweight='bold'
    )
    axes[2].axis('off')

    # panel 4: flood extent map
    axes[3].imshow(img_np)
    overlay = np.zeros((*flood_mask.shape, 4))
    overlay[flood_mask == 1] = [0.0, 0.4, 1.0, 0.55]
    axes[3].imshow(overlay)
    for i in range(1, grid):
        axes[3].axhline(i*tile_size, color='white', lw=1, alpha=0.4)
        axes[3].axvline(i*tile_size, color='white', lw=1, alpha=0.4)
    for tile_idx, (r, c) in enumerate(positions):
        pct   = tile_flood_pcts[tile_idx]
        color = '#0066ff' if pct > 20 else '#555555'
        axes[3].text(
            c*tile_size + tile_size//2,
            r*tile_size + tile_size//2,
            f'{pct}%',
            color='white', fontsize=13,
            ha='center', va='center',
            fontweight='bold',
            bbox=dict(
                boxstyle='round,pad=0.4',
                facecolor=color, alpha=0.75
            )
        )
    flood_p = mpatches.Patch(
        color='#0066ff', alpha=0.7, label='Flood zone'
    )
    dry_p = mpatches.Patch(
        color='#555555', alpha=0.5, label='Dry zone'
    )
    axes[3].legend(
        handles=[flood_p, dry_p],
        loc='lower right',
        facecolor='#111111',
        labelcolor='white',
        fontsize=9
    )
    axes[3].set_title(
        f'Stage 4: Flood extent map\nTotal coverage: {total_flood:.1f}%',
        color='white', fontsize=11, fontweight='bold'
    )
    axes[3].axis('off')

    fig.suptitle(
        'SynapseRecon  ·  Detect → Verify → Solve → Explain  |  WorldFloods UAV',
        color='white', fontsize=14, fontweight='bold',
        y=1.01
    )
    plt.tight_layout()

    fname = os.path.splitext(os.path.basename(image_path))[0]
    out   = f'/content/drive/MyDrive/SynapseRecon/demo/demo_{fname}.png'
    plt.savefig(
        out, dpi=150,
        bbox_inches='tight',
        facecolor='#0f0f0f'
    )
    plt.show()
    print(f"Saved: {out}")
    return out

In [39]:
test_dir   = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_test'
test_files = sorted(os.listdir(test_dir))

saved_demos = []
for f in test_files[:5]:
    print(f"\n{'='*40}")
    print(f"Processing: {f}")
    try:
        out = full_synapserecon_demo(
            os.path.join(test_dir, f),
            model,
            grid=2
        )
        saved_demos.append(out)
    except Exception as e:
        print(f"Error on {f}: {e}")
        continue

print(f"\nDone — {len(saved_demos)} demos saved")


Processing: 000000.jpg
Stage 1 — 4 fragments of interest detected
Model inference fallback: Eff_GAT.forward_with_feats() missing 2 required positional arguments: 'patch_feats' and 'batch'
Stage 2 — confidences: ['0.73', '0.85', '1.00', '0.88']
  Content pixels: 91296
  Water threshold: 153.0
  Flood pixels: 15778
Stage 4 — flood coverage: 6.0%
Saved: /content/drive/MyDrive/SynapseRecon/demo/demo_000000.png

Processing: 000001.jpg
Stage 1 — 4 fragments of interest detected
Model inference fallback: Eff_GAT.forward_with_feats() missing 2 required positional arguments: 'patch_feats' and 'batch'
Stage 2 — confidences: ['0.73', '1.00', '0.82', '0.98']
  Content pixels: 70455
  Water threshold: 153.0
  Flood pixels: 14150
Stage 4 — flood coverage: 5.4%
Saved: /content/drive/MyDrive/SynapseRecon/demo/demo_000001.png

Processing: 000002.jpg
Stage 1 — 4 fragments of interest detected
Model inference fallback: Eff_GAT.forward_with_feats() missing 2 required positional arguments: 'patch_feats' a

In [32]:
import numpy as np
from PIL import Image

test_dir = '/content/SynapseRecon/puzzle_diff/datasets/CelebA-HQ_test'
img_path = os.path.join(test_dir, '000001.jpg')

img_np = load_and_normalize(img_path, 512)

print(f"Shape: {img_np.shape}")
print(f"Min: {img_np.min()}, Max: {img_np.max()}")
print(f"Mean: {img_np.mean():.2f}")
print(f"Std: {img_np.std():.2f}")

gray = np.mean(img_np, axis=2)
print(f"\nGray percentiles:")
for p in [10, 20, 30, 40, 50, 60, 70]:
    print(f"  {p}%: {np.percentile(gray, p):.2f}")

# show unique value ranges
print(f"\nR channel: {img_np[:,:,0].min()}-{img_np[:,:,0].max()}")
print(f"G channel: {img_np[:,:,1].min()}-{img_np[:,:,1].max()}")
print(f"B channel: {img_np[:,:,2].min()}-{img_np[:,:,2].max()}")

Shape: (512, 512, 3)
Min: 0, Max: 255
Mean: 37.50
Std: 64.45

Gray percentiles:
  10%: 0.00
  20%: 0.00
  30%: 0.00
  40%: 0.00
  50%: 0.00
  60%: 0.00
  70%: 0.00

R channel: 0-255
G channel: 0-255
B channel: 0-255


In [40]:
from google.colab import files

for path in saved_demos:
    print(f"Downloading: {os.path.basename(path)}")
    files.download(path)

Downloading: demo_000000.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: demo_000001.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: demo_000004.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
# save current notebook to drive
!jupyter nbconvert --to notebook \
    /content/drive/MyDrive/SynapseRecon/SynapseRecon_Flood_Demo.ipynb \
    --output /content/drive/MyDrive/SynapseRecon/SynapseRecon_Flood_Demo.ipynb

[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/SynapseRecon/SynapseRecon_Flood_Demo.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent 

In [47]:
%cd /content/SynapseRecon

# configure git
!git config user.email "navami.navamitha@gmail.com"
!git config user.name "Navami"

# create branch
!git checkout -b flood-demo

# copy demo images and notebook from Drive
!mkdir -p demo_outputs
!cp /content/drive/MyDrive/SynapseRecon/demo/*.png demo_outputs/
!cp /content/drive/MyDrive/*.ipynb .

# check what's there
!ls *.ipynb
!ls demo_outputs/

/content/SynapseRecon
fatal: A branch named 'flood-demo' already exists.
GenAI.ipynb	 Untitled1.ipynb  Untitled4.ipynb  Untitled7.ipynb
nlp_lab.ipynb	 Untitled2.ipynb  Untitled5.ipynb  Untitled8.ipynb
Untitled0.ipynb  Untitled3.ipynb  Untitled6.ipynb
demo_000000.png  demo_000002.png  demo_000004.png
demo_000001.png  demo_000003.png


In [48]:
!git commit -m "Add SynapseRecon flood tile reassembly demo outputs"

# push to new branch — replace YOUR_TOKEN with your GitHub personal access token
!git push https://@github.com/Sevengods77/SynapseRecon.git flood-demo

On branch flood-demo
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   puzzle_diff/model/spatial_diffusion_on_angle.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	puzzle_diff/datasets/

no changes added to commit (use "git add" and/or "git commit -a")
Everything up-to-date
